In [ ]:
"""Scrape NFL coaching film data using the nflpro library and stores it.

It connects to a PostgreSQL database, retrieves play-by-play data and \
     schedule information, and then uses the NFLProAPI to fetch \
        coaching film metadata. Finally, it downloads the coaching \
            film using the nflpro.film module.

The script is organized into several sections (using cell delimiters "%%") \
    for clarity and modularity, making it suitable for interactive \
        execution in environments like Jupyter Notebook or VS Code's \
            Python Interactive window.

Key Functions:
- create_connection: Establishes a connection to the PostgreSQL database.
- execute_query: Executes a SQL query and returns the result.
- query_to_dataframe: Executes a SQL query and returns the result \
    as a Pandas DataFrame.
- write_dataframe_to_postgres: Writes a Pandas DataFrame to a PostgreSQL table.

Data Flow:
1. Connect to the PostgreSQL database.
2. Retrieve play-by-play data from the nflfastR_pbp table.
3. Retrieve schedule data from the nflfastR_schedule table.
4. Use the NFLProAPI to fetch coaching film metadata for each play.
5. Download the coaching film using the nflpro.film module.
"""

import logging
import os
from datetime import UTC, datetime

import nflpro
import pandas as pd
import psycopg2
from dotenv import load_dotenv


In [ ]:
config = {
    "host": "moose.local",
    # "port":"port",  # noqa: ERA001
    # "database":"database",  # noqa: ERA001
    # "username":"user",  # noqa: ERA001
    # "password":"pass",  # noqa: ERA001
}


def create_connection(
    config: dict[str, str] | None,
) -> psycopg2.extensions.connection | None:
    """Create a connection to the PostgreSQL database.

    Args:
        config (Dict[str, str]): A dictionary containing \
            the database configuration.
            Must include keys:
                - "host",
                - "database",
                - "username",
                - "password",
                - "port"

    Returns:
        A psycopg2 connection object if the connection is successful, \
            otherwise None.

    """
    load_dotenv()  # Load environment variables from .env file

    # Load configuration from environment variables as a base
    env_config = {
        "host": os.environ.get("DB_HOST"),
        "database": os.environ.get("DB_NAME"),
        "username": os.environ.get("DB_USER"),
        "password": os.environ.get("DB_PASS"),
        "port": os.environ.get("DB_PORT"),
    }

    # Override with values from the config dictionary if provided
    if config:
        env_config.update(config)

    logging.info(env_config)

    # Check for missing values
    if not all(env_config.values()):
        logging.error(
            "Not all database configuration values are set \
                (either in config or environment).",
        )
        return None  # Or raise an exception

    try:
        conn = psycopg2.connect(
            host=env_config["host"],
            database=env_config["database"],
            user=env_config["username"],
            password=env_config["password"],
            port=env_config["port"],
        )

    except psycopg2.Error as e:
        msg = f"Error connecting to database: {e}"
        logging.exception(msg)
        return None
    else:
        return conn


def execute_query(
    conn: psycopg2.extensions.connection,
    query: str,
    params: tuple | None = None,
) -> list[tuple[any, ...]] | None:
    """Execute a SQL query against the database.

    Args:
        conn (psycopg2.extensions.connection): A psycopg2 connection object.
        query (str): The SQL query to execute.
        params (tuple, optional):  The parameters to pass to \
            the query. Defaults to None

    Returns:
        A list of tuples containing the result rows, \
            or None if an error occurs.

    """
    cur = None  # Initialize cur outside the try block
    try:
        cur = conn.cursor()
        cur.execute(query, params)
        rows = cur.fetchall()
    except psycopg2.Error as e:
        msg = f"Error executing query: {e}"
        logging.exception(msg)
        return None
    else:
        return rows
    finally:
        if cur:  # Check if cur is defined before attempting to close
            cur.close()


def query_to_dataframe(
    conn: psycopg2.extensions.connection,
    query: str,
    params: tuple | None = None,
    print_query: bool | None = None,
) -> pd.DataFrame:
    """Execute a SQL query and returns the result as a Pandas DataFrame.

    Args:
        conn (psycopg2.extensions.connection): A psycopg2 connection object.
        query (str): The SQL query to execute.
        params (tuple, optional):  The parameters to pass to \
            the query. Defaults to None
        print_query (bool | none, optional): Prints the query if selected.
            Defaults to None

    Returns:
        A Pandas DataFrame of tuples containing the result \
            rows, or None if an error occurs.

    """
    rows = execute_query(conn, query, params)
    if rows is None:
        return pd.DataFrame()

    try:
        # Create a new cursor to get column names, without re-executing query
        cur = conn.cursor()
        cur.execute(query, params)
        if print_query:
            logging.info(query)
        colnames = [desc[0] for desc in cur.description]  # Get column names
    except psycopg2.Error as e:
        msg = f"Error getting column names: {e}"
        logging.exception(msg)
        return pd.DataFrame()
    finally:
        if cur:
            cur.close()

    return pd.DataFrame(rows, columns=colnames)



In [ ]:
def write_dataframe_to_postgres(
    df: pd.DataFrame,
    config: dict,
    table_name: str,
) -> None:
    """Write a DataFrame to PostgreSQL database table, overwriting if needed.

    Args:
        df (pd.DataFrame): The DataFrame to write.
        config (dict): Database connection configuration.
        table_name (str): The name of the table to write to.

    Returns:
        None

    Example:
    >>> config = {
    ...     "host": "localhost",
    ...     "database": "mydatabase",
    ...     "username": "myuser",
    ...     "password": "mypassword",
    ...     "port": "5432",
    ... }
    >>> write_dataframe_to_postgres(plays_df, config, "plays_with_schedule")

    """
    try:
        conn = create_connection(config=config)
        if conn is not None:
            {
                {
                    df.to_sql(
                        name=table_name,  # Table name in PostgreSQL
                        con=conn,  # Database connection
                        # Replace the table if it already exists
                        if_exists="replace",
                        # Do not write DataFrame index as a column
                        index=False,
                    ),
                },
            }
            msg = (
                "DataFrame written to PostgreSQL table"
                f"'{table_name}' successfully!",
            )
            logging.info(msg)
        else:
            logging.warning("Failed to create database connection.")
    except Exception as e:
        msg = f"Error writing to PostgreSQL: {e}"
        logging.exception(msg)
    finally:
        if conn:
            conn.close()



In [ ]:
schema_df = query_to_dataframe(
    conn=create_connection(config=config),
    query="""
    select *
    from "nflfastR_pbp"
    where season = (select max(season) from "nflfastR_pbp")
    limit 0
    """,
)

nflfastr_columns = schema_df.columns
nflfastr_columns  # noqa: B018


 ### Get schedule

In [ ]:
nflfastr_schedule_df = query_to_dataframe(
    conn=create_connection(config=config),
    query="""select
        game_id,
        old_game_id,
        season,
        week,
        game_type
    from "nflfastR_schedule"
    where season >= %s
    order by 1
    """,
    params=(2024,),
)

nflfastr_schedule_df = nflfastr_schedule_df.assign(
    week_slug=lambda x: x.apply(
        lambda x: nflpro.create_week_slug(
            week=x["week"],
            game_type=x["game_type"],
        ),
        axis=1,
    ),
)

nflfastr_schedule_df  # noqa: B018


 ### Get plays

In [ ]:
nflpro_api = nflpro.NFLProAPI()
# nflpro_api._refresh_token()  # noqa: ERA001


In [ ]:
today = datetime.now(UTC).strftime("%Y-%m-%d")

plays_df = query_to_dataframe(
    conn=create_connection(config=config),
    query="""select
        season,
        week,
        season_type,
        game_id,
        old_game_id,
        nfl_api_id,
        cast(play_id as varchar) as play_id,
        posteam,
        defteam,
        home_score,
        away_score,
        "desc",
        epa
    from "nflfastR_pbp"
    where season = (select max(season) from "nflfastR_pbp")
        and play = 1
        and posteam = %s
        and passer_player_name = %s
        and game_date < %s
    order by 1, epa desc
    limit 10
    """,
    params=(
        "PHI",
        "J.Hurts",
        today,
    ),
    print_query=True,
)

plays_df  # noqa: B018


In [ ]:
nflpro.download_film_from_dataframe(
    nfl_pro_api=nflpro_api,
    plays_df=plays_df,
    column_names={
        "game_id": "game_id",
        "old_game_id": "old_game_id",
        "nfl_api_id": "nfl_api_id",
        "posteam": "posteam",
        "play_id": "play_id",
    },
)
